# Alunos matriculados: separar o arquivo geral por DRE

Separa o `relatorio_matriculas_2026.csv` (rede inteira) em um arquivo por DRE e mostra o tamanho de cada um.
Cada escola importa a própria lista em **SisGI → Alunos do Refeitório → Importar planilha**, que aceita até **20 MB**.

**LGPD:** o arquivo tem CPF, NIS, endereço, telefones, pais etc. Por padrão os arquivos gerados levam **só as colunas que o SisGI usa**.
As células abaixo mostram apenas totais, nunca dados de alunos. Apague os arquivos gerados quando não forem mais necessários.

## 1. Configuração

In [1]:
import os, re, unicodedata
import pandas as pd

ARQUIVO = 'relatorio_matriculas_2026.csv'
PASTA_SAIDA = 'por_dre'
LIMITE_MB = 20                  # limite de envio da tela Alunos do Refeitório

# Colunas gravadas nos arquivos por DRE. O SisGI lê LotacaoID, Lotacao, AlunoID, Nome, RelacaoSerie e Turma;
# DRE e Municipio ficam só para conferência. Para gravar TODAS as colunas (não recomendado), use COLUNAS = None.
COLUNAS = ['DRE', 'Municipio', 'LotacaoID', 'Lotacao', 'RelacaoSerie', 'Turma', 'AlunoID', 'Nome']

MB = 1024 * 1024

## 2. Leitura do arquivo geral

In [2]:
df = pd.read_csv(ARQUIVO, sep=';', dtype=str, keep_default_na=False, encoding='utf-8')
print(f'{len(df):,} linhas, {df.shape[1]} colunas, {os.path.getsize(ARQUIVO) / MB:,.1f} MB'.replace(',', '.'))

faltando = [x for x in ['DRE', 'Municipio', 'LotacaoID', 'Lotacao', 'AlunoID', 'Nome', 'RelacaoSerie', 'Turma'] if x not in df.columns]
assert not faltando, f'Colunas faltando: {faltando}'

# Tamanho aproximado de cada linha com TODAS as colunas (para comparar com o arquivo reduzido)
# (texto de cada campo + 2 aspas + 1 separador por coluna)
df['_bytes'] = df.apply(lambda col: col.str.len()).sum(axis=1) + df.shape[1] * 3

313.647 linhas. 57 colunas. 277.8 MB


## 3. Arquivo reduzido (só as colunas usadas)

In [3]:
reduzido = df if COLUNAS is None else df[COLUNAS + ['_bytes']]
antes = len(reduzido)
# O SisGI guarda um registro por aluno em cada escola; repetidos (mais de uma matrícula na mesma escola) são descartados
reduzido = reduzido.drop_duplicates(subset=['LotacaoID', 'AlunoID'])
print(f'{antes - len(reduzido):,} linha(s) repetida(s) de aluno na mesma escola descartada(s)'.replace(',', '.'))
print(f'{reduzido["AlunoID"].nunique():,} alunos, {reduzido["LotacaoID"].nunique():,} escolas, {reduzido["DRE"].nunique()} DREs'.replace(',', '.'))

1.028 linha(s) repetida(s) de aluno na mesma escola descartada(s)
312.612 alunos. 631 escolas. 13 DREs


## 4. Gravar um arquivo por DRE

In [4]:
def nome_arquivo(texto):
    t = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode()
    t = re.sub(r'[^A-Za-z0-9]+', '_', t).strip('_').lower()
    return (t or 'sem_dre') + '.csv'

os.makedirs(PASTA_SAIDA, exist_ok=True)
linhas = []
for dre, g in reduzido.groupby('DRE', sort=True):
    caminho = os.path.join(PASTA_SAIDA, nome_arquivo(dre))
    g.drop(columns=['_bytes']).to_csv(caminho, sep=';', index=False, encoding='utf-8-sig', quoting=1, lineterminator='\r\n')
    linhas.append({
        'DRE': dre,
        'Municípios': g['Municipio'].nunique(),
        'Escolas': g['LotacaoID'].nunique(),
        'Alunos': len(g),
        'MB (arquivo gerado)': os.path.getsize(caminho) / MB,
        'MB (todas as colunas, aprox.)': g['_bytes'].sum() / MB,
        'Arquivo': caminho,
    })

resumo_dre = pd.DataFrame(linhas).sort_values('MB (arquivo gerado)', ascending=False).reset_index(drop=True)
total = resumo_dre.sum(numeric_only=True)
print(f'Total gerado: {total["MB (arquivo gerado)"]:.1f} MB em {len(resumo_dre)} arquivos '
      f'(com todas as colunas seriam ~{total["MB (todas as colunas, aprox.)"]:.1f} MB)')
resumo_dre.style.format({'MB (arquivo gerado)': '{:.2f}', 'MB (todas as colunas, aprox.)': '{:.2f}'})

Total gerado: 51.6 MB em 13 arquivos (com todas as colunas seriam ~273.2 MB)


,DRE,Municípios,Escolas,Alunos,MB (arquivo gerado),"MB (todas as colunas, aprox.)",Arquivo
0,DRE METROPOLITANA,11,138,85790,14.63,75.25,por_dre\dre_metropolitana.csv
1,DRE SINOP,18,58,43671,6.84,37.58,por_dre\dre_sinop.csv
2,DRE RONDONOPOLIS,14,60,32431,5.39,28.89,por_dre\dre_rondonopolis.csv
3,DRE BARRA DO GARCAS,19,58,20603,3.45,17.95,por_dre\dre_barra_do_garcas.csv
4,DRE TANGARA DA SERRA,7,39,19529,3.32,17.21,por_dre\dre_tangara_da_serra.csv
5,DRE PRIMAVERA DO LESTE,7,40,18642,3.23,16.43,por_dre\dre_primavera_do_leste.csv
6,DRE JUINA,10,48,17035,2.69,14.77,por_dre\dre_juina.csv
7,DRE DIAMANTINO,12,43,15177,2.52,13.36,por_dre\dre_diamantino.csv
8,DRE CACERES,12,37,15532,2.42,13.38,por_dre\dre_caceres.csv
9,DRE CONFRESA,13,35,11558,1.85,9.93,por_dre\dre_confresa.csv


## 5. Tamanho por escola

O que importa para a tela é o tamanho **por escola**. Se o arquivo da DRE passar de 20 MB, a escola não consegue enviá-lo.
A célula mostra quanto ocupa cada escola, tanto no formato reduzido quanto com todas as colunas.

In [5]:
por_escola = (reduzido.groupby(['DRE', 'Municipio', 'LotacaoID', 'Lotacao'])
              .agg(Alunos=('AlunoID', 'size'), bytes_completo=('_bytes', 'sum'))
              .reset_index())
# Tamanho reduzido estimado pela média de bytes por linha dos arquivos gerados
bytes_linha = total['MB (arquivo gerado)'] * MB / total['Alunos']
por_escola['MB (reduzido, aprox.)'] = por_escola['Alunos'] * bytes_linha / MB
por_escola['MB (todas as colunas, aprox.)'] = por_escola['bytes_completo'] / MB
por_escola = por_escola.drop(columns=['bytes_completo']).sort_values('Alunos', ascending=False)

print(f'Escolas: {len(por_escola)}; média de {por_escola["Alunos"].mean():.0f} alunos por escola; '
      f'maior escola: {por_escola["Alunos"].max()} alunos '
      f'({por_escola["MB (reduzido, aprox.)"].max():.2f} MB reduzido / {por_escola["MB (todas as colunas, aprox.)"].max():.2f} MB completo)')
print(f'DREs com arquivo acima de {LIMITE_MB} MB: {(resumo_dre["MB (arquivo gerado)"] > LIMITE_MB).sum()}')
por_escola.head(15).style.format({'MB (reduzido, aprox.)': '{:.2f}', 'MB (todas as colunas, aprox.)': '{:.2f}'})

Escolas: 631; média de 495 alunos por escola; maior escola: 2119 alunos (0.35 MB reduzido / 1.86 MB completo)
DREs com arquivo acima de 20 MB: 0


,DRE,Municipio,LotacaoID,Lotacao,Alunos,"MB (reduzido, aprox.)","MB (todas as colunas, aprox.)"
325,DRE METROPOLITANA,CUIABÁ,9547,EE PROFESSOR WELSON MESQUITA DE OLIVEIRA,2119,0.35,1.86
578,DRE SINOP,SINOP,91448,EE PROFESSORA EDELI MANTOVANI,1833,0.30,1.60
584,DRE SINOP,SORRISO,38598,EE IGNÁCIO SCHEVINSKI FILHO,1761,0.29,1.49
307,DRE METROPOLITANA,CUIABÁ,20516,EECM DR. MÁRIO DE CASTRO,1741,0.29,1.52
515,DRE RONDONOPOLIS,RONDONOPOLIS,11045,EECM PROFª ELIZABETH DE FREITAS MAGALHÃES,1713,0.28,1.56
298,DRE METROPOLITANA,CUIABÁ,10596,EE MILITAR DOM PEDRO II PRESIDENTE MÉDICI,1659,0.27,1.51
305,DRE METROPOLITANA,CUIABÁ,184152,EECM JUAREZ RODRIGUES DOS ANJOS,1571,0.26,1.36
182,DRE DIAMANTINO,NOVA MUTUM,145696,EECM VIRGÍLIO CORRÊA FILHO,1491,0.25,1.31
312,DRE METROPOLITANA,CUIABÁ,23329,EECM MALIK DIDIER NAMER ZAHAFI,1486,0.25,1.28
512,DRE RONDONOPOLIS,RONDONOPOLIS,10995,EE MAJOR OTAVIO PITALUGA,1476,0.24,1.31


## 6. Gravar um arquivo por escola

Cada escola recebe **só os próprios alunos**. Os arquivos ficam em `por_escola/<dre>/<LotacaoID>_<escola>.csv`,
com uma pasta por DRE para facilitar a entrega. O LotacaoID no começo do nome é o código da escola no SisGI.

In [6]:
PASTA_ESCOLAS = 'por_escola'

def nome_curto(texto, limite=60):
    t = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode()
    t = re.sub(r'[^A-Za-z0-9]+', '_', t).strip('_').lower()
    return t[:limite].rstrip('_')

linhas_escola = []
for (dre, lotacao_id), g in reduzido.groupby(['DRE', 'LotacaoID'], sort=True):
    pasta = os.path.join(PASTA_ESCOLAS, nome_arquivo(dre)[:-4])
    os.makedirs(pasta, exist_ok=True)
    escola = g['Lotacao'].iloc[0]
    caminho = os.path.join(pasta, f'{lotacao_id}_{nome_curto(escola)}.csv')
    g.drop(columns=['_bytes']).to_csv(caminho, sep=';', index=False, encoding='utf-8-sig', quoting=1, lineterminator='\r\n')
    linhas_escola.append({'DRE': dre, 'Municipio': g['Municipio'].iloc[0], 'LotacaoID': lotacao_id, 'Escola': escola,
                          'Alunos': len(g), 'KB': os.path.getsize(caminho) / 1024, 'Arquivo': caminho})

arquivos_escola = pd.DataFrame(linhas_escola)
print(f'{len(arquivos_escola)} arquivos em {arquivos_escola["DRE"].nunique()} pastas; '
      f'total {arquivos_escola["KB"].sum() / 1024:.1f} MB; maior {arquivos_escola["KB"].max():.0f} KB; '
      f'média {arquivos_escola["KB"].mean():.0f} KB')

# Resumo por DRE (quantos arquivos em cada pasta)
(arquivos_escola.groupby('DRE')
    .agg(Escolas=('LotacaoID', 'size'), Alunos=('Alunos', 'sum'), MB=('KB', lambda s: s.sum() / 1024), Maior_KB=('KB', 'max'))
    .sort_values('Alunos', ascending=False)
    .style.format({'MB': '{:.2f}', 'Maior_KB': '{:.0f}'}))

631 arquivos em 13 pastas; total 51.7 MB; maior 397 KB; média 84 KB


,Escolas,Alunos,MB,Maior_KB
DRE,,,,
DRE METROPOLITANA,138,85790,14.64,397
DRE SINOP,58,43671,6.84,293
DRE RONDONOPOLIS,60,32431,5.39,319
DRE BARRA DO GARCAS,58,20603,3.46,186
DRE TANGARA DA SERRA,39,19529,3.33,229
DRE PRIMAVERA DO LESTE,40,18642,3.23,218
DRE JUINA,48,17035,2.70,187
DRE CACERES,37,15532,2.42,180
DRE DIAMANTINO,43,15177,2.53,258


In [7]:
# Índice das escolas (sem dados de alunos), útil para conferir a entrega
arquivos_escola.drop(columns=['Arquivo']).to_csv(os.path.join(PASTA_ESCOLAS, 'indice_escolas.csv'),
                                               sep=';', index=False, encoding='utf-8-sig')
print('Índice gravado em', os.path.join(PASTA_ESCOLAS, 'indice_escolas.csv'))

Índice gravado em por_escola\indice_escolas.csv


## 7. Depois de usar

- Entregue a cada DRE só a pasta ou o arquivo dela, e a cada escola só o arquivo dela.
- Apague as pastas `por_dre` e `por_escola` e o arquivo geral quando terminar. Eles têm nomes e códigos de alunos.